# Utilidades
Funciones que se usan en todos los notebooks del proceso. Se cargan con `%run ./00_utils`.

In [ ]:
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CAPAS = ["bronze", "silver", "golden"]

In [ ]:
# Imprime un mensaje con fecha, nivel y entidad
def log(mensaje, nivel="INFO", entidad="-"):
    print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] [{nivel}] [{entidad}] {mensaje}")


# Lee los parametros del notebook (catalogo y storage)
def obtener_parametros():
    dbutils.widgets.text("catalogo", "catalog_dev")
    dbutils.widgets.text("storage_name", "")
    catalogo = dbutils.widgets.get("catalogo").strip()
    storage = dbutils.widgets.get("storage_name").strip()
    if not catalogo or not storage:
        raise ValueError("Los parametros 'catalogo' y 'storage_name' son obligatorios")
    return {"catalogo": catalogo, "storage": storage}


def ruta_contenedor(storage, contenedor):
    return f"abfss://{contenedor}@{storage}.dfs.core.windows.net"


# Carpeta de la capa raw donde se encuentran los csv de cada entidad
def ruta_raw(storage, entidad):
    return f"{ruta_contenedor(storage, 'raw')}/instacart/{entidad}/"


# Ruta fisica de una tabla: <capa>/<catalogo>/<tabla>
def ruta_tabla(storage, capa, catalogo, tabla):
    return f"{ruta_contenedor(storage, capa)}/{catalogo}/{tabla}"


def validar_no_vacio(df, nombre):
    total = df.count()
    if total == 0:
        raise ValueError(f"El dataframe {nombre} esta vacio")
    log(f"{total:,} registros", "INFO", nombre)
    return total


# insertInto inserta por posicion, por eso primero se ordenan las columnas como en la tabla destino
def escribir_tabla(df, tabla, modo="overwrite"):
    columnas_destino = spark.table(tabla).columns
    faltantes = set(columnas_destino) - set(df.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas para {tabla}: {sorted(faltantes)}")
    df.select(*columnas_destino).write.mode(modo).insertInto(tabla)
    log(f"Escritura {modo} completada", "INFO", tabla)


# Merge: actualiza los registros que ya existen e inserta los nuevos
def merge_delta(df, tabla, llaves):
    columnas_destino = spark.table(tabla).columns
    df = df.select(*columnas_destino).dropDuplicates(llaves)
    condicion = " AND ".join([f"t.{k} = s.{k}" for k in llaves])
    (DeltaTable.forName(spark, tabla).alias("t")
        .merge(df.alias("s"), condicion)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    log("Merge completado", "INFO", tabla)


# Columnas de auditoria de la capa bronze
def agregar_auditoria(df):
    return (df.withColumn("ingestion_date", F.current_timestamp())
              .withColumn("source_file", F.col("_metadata.file_path")))